# 01 - مقدمة في التضمينات الشعاعية وقواعد بيانات المتجهات
## (Introduction to Vector Embeddings & Vector Databases)

يركز هذا الدفتر على تطبيق كافة المفاهيم المطروحة في المحاضرة 19 من الكورس:
1. **المطابقة الحرفية (Exact Match)** في قواعد البيانات التقليدية مقابل **المطابقة الدلالية (Semantic Match)** في قواعد بيانات المتجهات.
2. تمثيل النصوص كمتجهات خصائص (**Feature Vectors**) عبر مثال أفلام مارفل (*Iron Man*, *Hulk*, *Sherlock Holmes*).
3. الحساب الرياضي الدقيق لتشابه جيب التمام (**Cosine Similarity**) خطوة بخطوة من الصفر، وبواسطة **NumPy** المتجهي.
4. التصور الهندسي والرسم البياني ثنائي الأبعاد (**2D Feature Plotting**) للمتجهات باستخدام **Matplotlib**.
5. تجربة عملية مع نموذج تضمين حقيقي (**Sentence-Transformers**) لإثبات تقارب الكلمات دلالياً.

---
### 1. المطابقة الحرفية (Traditional DB) مقابل المطابقة الدلالية (Vector DB)
في قواعد البيانات التقليدية مثل SQL، نبحث عن تطابق دقيق للكلمات (`WHERE word = "cat"`). بينما في قواعد بيانات المتجهات نبحث عن تقارب المعنى في الفضاء الشعاعي.

In [ ]:
# محاكاة البيانات المخزنة في قاعدة البيانات
database_records = ["cat", "dog", "kitten", "puppy"]
search_query = "cat"

# 1. سلوك قاعدة البيانات التقليدية (Exact Match)
traditional_results = [record for record in database_records if record.lower() == search_query.lower()]
print(f"🔍 Traditional DB Search for '{search_query}':")
print(f"   Results: {traditional_results}")
print("   ⚠️ لاحظ: تم استبعاد 'kitten' تماماً رغم أنها تعني قطة صغيرة ولها نفس المعنى!")

---
### 2. تمثيل فضاء الخصائص: مثال أفلام مارفل (The Feature Space)
نفترض فضاء خصائص مبسطاً من 3 أبعاد:
* **البعد 0 (Action):** نسبة الأكشن والإثارة (0.0 -> 1.0)
* **البعد 1 (Comedy):** نسبة الكوميديا والضحك (0.0 -> 1.0)
* **البعد 2 (Suspense):** نسبة الغموض والتشويق (0.0 -> 1.0)

In [ ]:
import numpy as np

# تعريف متجهات الأفلام بناءً على الخصائص الثلاث: [Action, Comedy, Suspense]
movies = {
    "Iron Man": np.array([0.95, 0.20, 0.60]),
    "Hulk": np.array([0.96, 0.40, 0.70]),
    "Sherlock Holmes": np.array([0.60, 0.75, 0.90])
}

for title, vector in movies.items():
    print(f"🎬 {title:16} -> Vector: {vector} | Shape: {vector.shape}")

---
### 3. الحساب الرياضي لتشابه جيب التمام (Cosine Similarity)
المعادلة الرياضية الأساسية لقياس التشابه الزاوي بين متجهين:
$$\text{Cosine Similarity}(\mathbf{A}, \mathbf{B}) = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^n A_i B_i}{\sqrt{\sum_{i=1}^n A_i^2} \cdot \sqrt{\sum_{i=1}^n B_i^2}}$$

In [ ]:
import math

# 1. الدالة الحسابية من الصفر (Pure Python)
def cosine_similarity_scratch(vec_a, vec_b):
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a ** 2 for a in vec_a))
    norm_b = math.sqrt(sum(b ** 2 for b in vec_b))
    return dot_product / (norm_a * norm_b)

# 2. الدالة المتجهية السريعة عبر NumPy
def cosine_similarity_numpy(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

# حساب التشابه بين Iron Man وباقي الأفلام
sim_iron_hulk = cosine_similarity_numpy(movies["Iron Man"], movies["Hulk"])
sim_iron_sherlock = cosine_similarity_numpy(movies["Iron Man"], movies["Sherlock Holmes"])

print(f"✨ Cosine Similarity (Iron Man vs Hulk):            {sim_iron_hulk:.4f}")
print(f"✨ Cosine Similarity (Iron Man vs Sherlock Holmes): {sim_iron_sherlock:.4f}")

# منطق نظام التوصية والبحث الدلالي
recommended = "Hulk" if sim_iron_hulk > sim_iron_sherlock else "Sherlock Holmes"
print(f"\n🍿 اقتراح النظام: بعد مشاهدة Iron Man، يتم ترشيح: '{recommended}' بنسبة تطابق {sim_iron_hulk * 100:.1f}%")

In [ ]:
# حساب مصفوفة التشابه الكاملة (All-to-All Cosine Similarity Matrix)
def compute_similarity_matrix(vectors_dict):
    names = list(vectors_dict.keys())
    mat = np.array(list(vectors_dict.values()))
    dot_matrix = np.dot(mat, mat.T)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    return names, dot_matrix / (norms * norms.T)

names, sim_matrix = compute_similarity_matrix(movies)

print("📊 مصفوفة التشابه الدلالي بين جميع الأفلام:")
print(f"                 {names[0]:15}  {names[1]:15}  {names[2]:15}")
for idx, row in enumerate(sim_matrix):
    print(f"{names[idx]:16}: {row[0]:.4f}           {row[1]:.4f}           {row[2]:.4f}")

---
### 4. التصور البصري ثنائي الأبعاد (2D Geometric View)
نسقط المتجهات على بعدي الأكشن (Action) والكوميديا (Comedy) لتوضيح قرب الزاوية والمسافة هندسياً.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
colors = {"Iron Man": "#e74c3c", "Hulk": "#27ae60", "Sherlock Holmes": "#2980b9"}

for name, vec in movies.items():
    action, comedy = vec[0], vec[1]
    ax.scatter(action, comedy, color=colors[name], s=180, label=name, zorder=5)
    ax.arrow(0, 0, action, comedy, color=colors[name], alpha=0.35, width=0.006, head_width=0.025, length_includes_head=True)
    ax.annotate(f"{name}\n({action:.2f}, {comedy:.2f})", (action + 0.02, comedy + 0.02), fontsize=10, fontweight="bold", color=colors[name])

ax.set_title("Movie Embeddings in 2D Feature Space (Action vs. Comedy)", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Action Feature Score (0.0 -> 1.0)", fontsize=11)
ax.set_ylabel("Comedy Feature Score (0.0 -> 1.0)", fontsize=11)
ax.set_xlim(0, 1.2)
ax.set_ylim(0, 1.0)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()

---
### 5. تجربة عملية مع نموذج تضمين حقيقي (Sentence-Transformers)
نستخدم نموذج `all-MiniLM-L6-v2` المجاني ومفتوح المصدر لاختبار الكلمات التي ذكرها المحاضر (`cat`, `kitten`, `dog`, `puppy`, `automobile`).

In [ ]:
from sentence_transformers import SentenceTransformer

# تحميل النموذج وتوليد المتجهات الحقيقية
print("⏳ جارٍ تحميل نموذج all-MiniLM-L6-v2...")
model = SentenceTransformer("all-MiniLM-L6-v2")

words = ["cat", "kitten", "puppy", "dog", "automobile"]
embeddings = model.encode(words)

print(f"✅ تم توليد التضمينات بنجاح!")
print(f"📏 أبعاد كل متجه: {embeddings.shape[1]} بُعداً (Embedding Dimensions)\n")

# حساب تشابه الاستعلام "cat" مع باقي الكلمات باستخدام NumPy المتجهي
query_vec = model.encode(["cat"])
query_norm = np.linalg.norm(query_vec)
emb_norms = np.linalg.norm(embeddings, axis=1)
similarities = np.dot(query_vec, embeddings.T)[0] / (query_norm * emb_norms)

print("🔍 نتائج البحث الدلالي عن كلمة 'cat':")
for word, score in sorted(zip(words, similarities), key=lambda x: x[1], reverse=True):
    bar = "█" * int(score * 30)
    print(f"   {word:12} | Cosine Similarity: {score:.4f}  {bar}")

### 💡 الخلاصة والاستنتاج (Key Takeaway):
1. كلمة `kitten` حصلت على أعلى نسبة تشابه دلالي مع `cat` (أكثر من 0.75)، على الرغم من اختلاف الحروف تماماً!
2. كلمتا `dog` و `puppy` جاءتا في مرتبة تالية لارتباطهما بفئة الحيوانات الأليفة المنزلية.
3. كلمة `automobile` جاءت في المؤخرة بتشابه ضئيل جداً لاختلاف الفئة الدلالية تماماً.
وهذا يثبت عملياً الأساس الذي تُبنى عليه كافة أنظمة الـ **RAG الحديثة**.